In [1]:
!pip install pyspark

In [2]:
from google.colab import files
from pyspark.sql import SparkSession, functions as F
from pyspark.sql.window import Window
from pyspark import StorageLevel
uploaded = files.upload()
spark = SparkSession.builder.appName("EnergyAssessment").getOrCreate()

Saving maintenance_tickets_1600.csv to maintenance_tickets_1600.csv
Saving meter_master.csv to meter_master.csv
Saving meter_readings_90000.csv to meter_readings_90000.csv
Saving meter_readings_extra_12000.csv to meter_readings_extra_12000.csv


In [3]:
uploaded = files.upload()

Saving substation_master.csv to substation_master.csv
Saving tariff_master.csv to tariff_master.csv


In [4]:
def load_csv(filename):
    return spark.read.csv(filename, header=True, inferSchema=True)
readings_df = load_csv("meter_readings_90000.csv")


In [5]:
extra_readings_df = load_csv("meter_readings_extra_12000.csv")

In [6]:
substation_df=load_csv("substation_master.csv")
tariff_df=load_csv("tariff_master.csv")
meter_df=load_csv("meter_master.csv")
maintenance_df=load_csv("maintenance_tickets_1600.csv")

In [7]:
readings_df.printSchema()
extra_readings_df.printSchema()
substation_df.printSchema()
tariff_df.printSchema()
meter_df.printSchema()
maintenance_df.printSchema()

root
 |-- reading_id: string (nullable = true)
 |-- meter_id: string (nullable = true)
 |-- reading_timestamp: timestamp (nullable = true)
 |-- energy_kwh: double (nullable = true)
 |-- voltage_v: double (nullable = true)
 |-- power_factor: double (nullable = true)
 |-- outage_minutes: integer (nullable = true)
 |-- reading_status: string (nullable = true)
 |-- source_type: string (nullable = true)
 |-- ambient_temperature_c: double (nullable = true)
 |-- operator_id: string (nullable = true)

root
 |-- reading_id: string (nullable = true)
 |-- meter_id: string (nullable = true)
 |-- reading_timestamp: timestamp (nullable = true)
 |-- energy_kwh: double (nullable = true)
 |-- voltage_v: double (nullable = true)
 |-- power_factor: double (nullable = true)
 |-- outage_minutes: integer (nullable = true)
 |-- reading_status: string (nullable = true)
 |-- source_type: string (nullable = true)
 |-- ambient_temperature_c: double (nullable = true)
 |-- operator_id: string (nullable = true)

ro

In [8]:
 print(readings_df.count())
 readings_df.show(10)

90000
+----------+--------+-------------------+----------+---------+------------+--------------+--------------+-----------+---------------------+-----------+
|reading_id|meter_id|  reading_timestamp|energy_kwh|voltage_v|power_factor|outage_minutes|reading_status|source_type|ambient_temperature_c|operator_id|
+----------+--------+-------------------+----------+---------+------------+--------------+--------------+-----------+---------------------+-----------+
| RD0000001|   M0082|2026-04-01 02:00:00|     22.53|    213.1|       0.939|             0|         Valid| SmartMeter|                 36.2|      OP029|
| RD0000002|   M0020|2026-01-31 06:00:00|      2.97|    227.4|       0.845|             0|         Valid| SmartMeter|                 41.2|      OP025|
| RD0000003|   M0054|2026-04-11 10:15:00|      8.61|    239.3|       0.879|             0|     Estimated| SmartMeter|                 20.9|      OP007|
| RD0000004|   M0111|2026-03-14 00:00:00|      1.87|    246.5|       0.831|       

In [9]:
temp_df=readings_df.select(
    "reading_id",
    "meter_id",
    "reading_timestamp",
    "energy_kwh",
    "reading_status"
).show()

+----------+--------+-------------------+----------+--------------+
|reading_id|meter_id|  reading_timestamp|energy_kwh|reading_status|
+----------+--------+-------------------+----------+--------------+
| RD0000001|   M0082|2026-04-01 02:00:00|     22.53|         Valid|
| RD0000002|   M0020|2026-01-31 06:00:00|      2.97|         Valid|
| RD0000003|   M0054|2026-04-11 10:15:00|      8.61|     Estimated|
| RD0000004|   M0111|2026-03-14 00:00:00|      1.87|         Valid|
| RD0000005|   M0135|2026-05-27 01:45:00|      4.39|         Valid|
| RD0000006|   M0139|2026-02-20 02:15:00|      0.36|         Valid|
| RD0000007|   M0142|2026-02-01 05:30:00|     79.47|         Valid|
| RD0000008|   M0126|2026-03-01 20:45:00|     28.35|      Rejected|
| RD0000009|   M0005|2026-05-03 10:30:00|      1.96|     Estimated|
| RD0000010|   M0052|2026-04-10 05:45:00|     22.52|         Valid|
| RD0000011|   M0137|2026-02-03 16:00:00|     10.51|         Valid|
| RD0000012|   M0009|2026-05-01 00:00:00|      1

In [10]:
readings_df=readings_df.withColumnRenamed("source_type","collection_mode")

In [11]:
from pyspark.sql.functions import col,desc
df=readings_df.filter(
    (col("reading_status")=="Valid")&
    (col("energy_kwh") > 20)
)
df.sort(desc("energy_kwh")).show(1)

+----------+--------+-------------------+----------+---------+------------+--------------+--------------+---------------+---------------------+-----------+
|reading_id|meter_id|  reading_timestamp|energy_kwh|voltage_v|power_factor|outage_minutes|reading_status|collection_mode|ambient_temperature_c|operator_id|
+----------+--------+-------------------+----------+---------+------------+--------------+--------------+---------------+---------------------+-----------+
| RD0041703|   M0101|2026-03-06 22:30:00|     110.0|    213.1|       0.822|             0|         Valid|     SmartMeter|                 31.1|      OP010|
+----------+--------+-------------------+----------+---------+------------+--------------+--------------+---------------+---------------------+-----------+
only showing top 1 row


In [12]:
from pyspark.sql.functions import col, when

df = readings_df.withColumn(
    "energy_band",
    when(col("energy_kwh").isNull(), "Unknown")
    .when(col("energy_kwh") < 5, "Low")
    .when((col("energy_kwh") >= 5) & (col("energy_kwh") <= 25), "Medium")
    .otherwise("High")
)

df.select("energy_kwh", "energy_band").show()

+----------+-----------+
|energy_kwh|energy_band|
+----------+-----------+
|     22.53|     Medium|
|      2.97|        Low|
|      8.61|     Medium|
|      1.87|        Low|
|      4.39|        Low|
|      0.36|        Low|
|     79.47|       High|
|     28.35|       High|
|      1.96|        Low|
|     22.52|     Medium|
|     10.51|     Medium|
|      1.49|        Low|
|     15.75|     Medium|
|     15.96|     Medium|
|      6.06|     Medium|
|     25.82|       High|
|      6.23|     Medium|
|      3.66|        Low|
|     104.8|       High|
|      1.35|        Low|
+----------+-----------+
only showing top 20 rows


In [13]:
from pyspark.sql.functions import col

null_count1 = readings_df.filter(
    col("energy_kwh").isNull()
).count()

print("Null count for energy_kwh:", null_count1)
null_count2=readings_df.filter(
    col("voltage_v").isNull()
).count()
print("Null count for voltage_v:", null_count2)

null_count3=readings_df.filter(
    col("power_factor").isNull()
).count()
print("Null count for power_factor:", null_count3)

null_count4=readings_df.filter(
    col("operator_id").isNull()
).count()
print("Null count for operator_id:", null_count4)

Null count for energy_kwh: 1620
Null count for voltage_v: 2070
Null count for power_factor: 1440
Null count for operator_id: 5400


In [14]:
filled_df = readings_df.na.fill({
    "operator_id": "UNASSIGNED",
    "power_factor": 0.90
})

filled_df.show()

+----------+--------+-------------------+----------+---------+------------+--------------+--------------+---------------+---------------------+-----------+
|reading_id|meter_id|  reading_timestamp|energy_kwh|voltage_v|power_factor|outage_minutes|reading_status|collection_mode|ambient_temperature_c|operator_id|
+----------+--------+-------------------+----------+---------+------------+--------------+--------------+---------------+---------------------+-----------+
| RD0000001|   M0082|2026-04-01 02:00:00|     22.53|    213.1|       0.939|             0|         Valid|     SmartMeter|                 36.2|      OP029|
| RD0000002|   M0020|2026-01-31 06:00:00|      2.97|    227.4|       0.845|             0|         Valid|     SmartMeter|                 41.2|      OP025|
| RD0000003|   M0054|2026-04-11 10:15:00|      8.61|    239.3|       0.879|             0|     Estimated|     SmartMeter|                 20.9|      OP007|
| RD0000004|   M0111|2026-03-14 00:00:00|      1.87|    246.5|  

In [15]:
filled_df.select("operator_id", "power_factor").show()

+-----------+------------+
|operator_id|power_factor|
+-----------+------------+
|      OP029|       0.939|
|      OP025|       0.845|
|      OP007|       0.879|
| UNASSIGNED|       0.831|
|      OP019|       0.966|
|      OP021|       0.788|
|      OP016|       0.986|
|      OP025|       0.818|
|      OP010|       0.817|
|      OP028|       0.804|
|      OP030|         0.9|
|      OP027|        0.81|
|      OP005|       0.807|
|      OP019|       0.884|
|      OP036|       0.919|
|      OP035|       0.895|
|      OP003|       0.762|
|      OP021|       0.914|
|      OP014|       0.953|
|      OP020|       0.771|
+-----------+------------+
only showing top 20 rows


In [20]:
df=readings_df.filter(
    col("energy_kwh").isNull()
).count()
print("Null count for energy_kwh:", df)
df=readings_df.dropna(
    subset=["energy_kwh"]
).count()
print("notNull count for energy_kwh:", df)

Null count for energy_kwh: 1620
notNull count for energy_kwh: 88380


In [22]:
from pyspark.sql.functions import col, sum

joined_df = readings_df.join(
    meter_df,
    on="meter_id",
    how="inner"
)

result_df = joined_df.filter(
    col("reading_status") == "Valid"
).groupBy(
    "site_type"
).agg(
    sum("energy_kwh").alias("total_energy")
)

result_df.show()

+---------------+-----------------+
|      site_type|     total_energy|
+---------------+-----------------+
|Public Facility|        138870.41|
|    Residential|100863.2900000004|
|     Industrial|710317.4100000021|
|     Commercial|451525.1799999996|
+---------------+-----------------+



In [23]:
from pyspark.sql.functions import col, sum

result_df = readings_df \
    .join(meter_df, on="meter_id", how="inner") \
    .join(substation_df, on="substation_id", how="inner") \
    .filter(col("reading_status") == "Valid") \
    .groupBy("region", "city") \
    .agg(
        sum("energy_kwh").alias("total_valid_energy")
    ) \
    .orderBy("region", "city")

result_df.show()

+-------+-------------+------------------+
| region|         city|total_valid_energy|
+-------+-------------+------------------+
|Central|       Indore|176503.91000000012|
|  North|        Delhi| 94234.89000000014|
|  North|       Jaipur| 47863.15999999996|
|  North|      Lucknow| 142610.3999999998|
|  South|    Bengaluru| 73787.59999999998|
|  South|      Chennai| 51829.05999999996|
|  South|    Hyderabad|51937.789999999884|
|  South|        Kochi| 204272.4600000002|
|  South|Visakhapatnam|205015.82999999932|
|   West|    Ahmedabad| 138627.7400000004|
|   West|       Mumbai| 52116.20999999999|
|   West|         Pune|162777.23999999996|
+-------+-------------+------------------+



In [24]:
from pyspark.sql.functions import col

energy_charge_df = readings_df \
    .join(meter_df, on="meter_id", how="inner") \
    .join(tariff_df, on="tariff_id", how="inner") \
    .filter(
        (col("reading_status") != "Rejected") &
        (col("energy_kwh").isNotNull())
    ) \
    .withColumn(
        "energy_charge_inr",
        col("energy_kwh") * col("rate_per_kwh_inr")
    )

energy_charge_df.select(
    "meter_id",
    "reading_status",
    "energy_kwh",
    "rate_per_kwh_inr",
    "energy_charge_inr"
).show()

+--------+--------------+----------+----------------+------------------+
|meter_id|reading_status|energy_kwh|rate_per_kwh_inr| energy_charge_inr|
+--------+--------------+----------+----------------+------------------+
|   M0082|         Valid|     22.53|             6.7|150.95100000000002|
|   M0020|         Valid|      2.97|             6.4|19.008000000000003|
|   M0054|     Estimated|      8.61|             8.2| 70.60199999999999|
|   M0111|         Valid|      1.87|             5.1| 9.536999999999999|
|   M0135|         Valid|      4.39|             8.2|            35.998|
|   M0139|         Valid|      0.36|             6.4|             2.304|
|   M0142|         Valid|     79.47|             9.1| 723.1769999999999|
|   M0005|     Estimated|      1.96|             5.1| 9.995999999999999|
|   M0052|         Valid|     22.52|             6.7|150.88400000000001|
|   M0137|         Valid|     10.51|             6.7|            70.417|
|   M0009|     Estimated|      1.49|             6.

In [25]:
 from pyspark.sql.functions import count, avg

result_df = maintenance_df \
    .join(meter_df, on="meter_id", how="inner") \
    .groupBy("site_type", "severity") \
    .agg(
        count("*").alias("ticket_count"),
        avg("maintenance_cost_inr").alias("avg_maintenance_cost_inr")
    ) \
    .orderBy("site_type", "severity")

result_df.show()

+---------------+--------+------------+------------------------+
|      site_type|severity|ticket_count|avg_maintenance_cost_inr|
+---------------+--------+------------+------------------------+
|     Commercial|Critical|          37|      13424.726216216219|
|     Commercial|    High|         130|      12293.073793103451|
|     Commercial|     Low|         166|      12223.517692307689|
|     Commercial|  Medium|         171|      13706.600886075952|
|     Industrial|Critical|          17|      12296.198000000002|
|     Industrial|    High|          63|      12838.869344262299|
|     Industrial|     Low|          66|       12717.52508474576|
|     Industrial|  Medium|         106|      12043.898526315788|
|Public Facility|Critical|          15|       8986.843571428572|
|Public Facility|    High|          48|      12905.279347826086|
|Public Facility|     Low|          60|      11195.929464285715|
|Public Facility|  Medium|          97|      13050.628421052634|
|    Residential|Critical

In [27]:
from pyspark.sql.functions import avg, sum, desc

result_df = readings_df \
    .join(meter_df, on="meter_id", how="inner") \
    .join(substation_df, on="substation_id", how="inner") \
    .groupBy("substation_id", "city") \
    .agg(\
        avg("voltage_v").alias("avg_voltage_v"), \
        sum("outage_minutes").alias("total_outage_minutes") \
    ) \
    .orderBy(desc("total_outage_minutes"))

result_df.show(5, truncate=False)

+-------------+-------------+------------------+--------------------+
|substation_id|city         |avg_voltage_v     |total_outage_minutes|
+-------------+-------------+------------------+--------------------+
|SUB10        |Visakhapatnam|230.04010567581344|91224               |
|SUB09        |Kochi        |230.04586272640606|81734               |
|SUB04        |Pune         |230.0558687466385 |75872               |
|SUB08        |Ahmedabad    |229.88012557077673|75581               |
|SUB03        |Chennai      |230.03094896227708|61662               |
+-------------+-------------+------------------+--------------------+
only showing top 5 rows


In [28]:
result_df = readings_df \
    .groupBy("collection_mode", "reading_status") \
    .agg(
        count("*").alias("reading_count")
    ) \
    .filter(
        col("reading_status").isin(
            "Valid",
            "Estimated",
            "Rejected"
        )
    ) \
    .orderBy(col("reading_count").desc())

result_df.show()

+---------------+--------------+-------------+
|collection_mode|reading_status|reading_count|
+---------------+--------------+-------------+
|     SmartMeter|         Valid|        65521|
|     SmartMeter|     Estimated|        10443|
|         Manual|         Valid|         8965|
|     SmartMeter|      Rejected|         3202|
|         Manual|     Estimated|         1405|
|         Manual|      Rejected|          464|
+---------------+--------------+-------------+



In [29]:
from pyspark.sql.functions import col, to_timestamp, year, month, sum

result_df = readings_df \
    .withColumn(
        "reading_timestamp",
        to_timestamp("reading_timestamp")
    ) \
    .withColumn("year", year("reading_timestamp")) \
    .withColumn("month", month("reading_timestamp")) \
    .join(meter_df, on="meter_id", how="inner") \
    .join(substation_df, on="substation_id", how="inner") \
    .filter(col("reading_status") == "Valid") \
    .groupBy("region", "year", "month") \
    .agg(
        sum("energy_kwh").alias("monthly_valid_energy")
    ) \
    .orderBy("region", "year", "month")

result_df.show()

+-------+----+-----+--------------------+
| region|year|month|monthly_valid_energy|
+-------+----+-----+--------------------+
|Central|2026|    1|   31475.57999999999|
|Central|2026|    2|  28161.410000000007|
|Central|2026|    3|            29805.85|
|Central|2026|    4|  29475.339999999982|
|Central|2026|    5|  30048.879999999986|
|Central|2026|    6|   27536.84999999999|
|  North|2026|    1|  48849.309999999925|
|  North|2026|    2|  45444.249999999985|
|  North|2026|    3|   51094.61000000006|
|  North|2026|    4|   43451.02000000005|
|  North|2026|    5|   48796.93000000004|
|  North|2026|    6|   47072.32999999992|
|  South|2026|    1|   97547.38999999987|
|  South|2026|    2|   90864.80000000006|
|  South|2026|    3|  102258.77000000005|
|  South|2026|    4|   97266.94000000005|
|  South|2026|    5|   101204.8699999999|
|  South|2026|    6|   97699.97000000002|
|   West|2026|    1|   59967.56999999997|
|   West|2026|    2|   55807.89999999989|
+-------+----+-----+--------------

In [30]:
from pyspark.sql.functions import col, sum, dense_rank
from pyspark.sql.window import Window

meter_energy_df = readings_df \
    .join(meter_df, on="meter_id", how="inner") \
    .filter(col("reading_status") == "Valid") \
    .groupBy("substation_id", "meter_id") \
    .agg(
        sum("energy_kwh").alias("total_valid_energy")
    )

window_spec = Window.partitionBy("substation_id") \
                    .orderBy(col("total_valid_energy").desc())

result_df = meter_energy_df.withColumn(
    "rank",
    dense_rank().over(window_spec)
).filter(
    col("rank") <= 3
)

result_df.show()

+-------------+--------+------------------+----+
|substation_id|meter_id|total_valid_energy|rank|
+-------------+--------+------------------+----+
|        SUB01|   M0130|           9364.98|   1|
|        SUB01|   M0110| 9259.750000000002|   2|
|        SUB01|   M0068| 8832.580000000005|   3|
|        SUB02|   M0071|          29242.04|   1|
|        SUB02|   M0145|28641.319999999992|   2|
|        SUB02|   M0133| 7060.140000000003|   3|
|        SUB03|   M0096| 9632.740000000003|   1|
|        SUB03|   M0080| 9525.940000000002|   2|
|        SUB03|   M0073| 9369.789999999995|   3|
|        SUB04|   M0038|          29505.88|   1|
|        SUB04|   M0099|29216.319999999996|   2|
|        SUB04|   M0098|28613.460000000014|   3|
|        SUB05|   M0094| 9659.639999999996|   1|
|        SUB05|   M0050| 9570.859999999997|   2|
|        SUB05|   M0008| 9524.780000000004|   3|
|        SUB06|   M0126|32573.380000000023|   1|
|        SUB06|   M0128|30748.840000000004|   2|
|        SUB06|   M0

In [31]:
from pyspark.sql.functions import col, sum, to_timestamp
from pyspark.sql.window import Window

valid_df = readings_df \
    .filter(col("reading_status") == "Valid") \
    .withColumn(
        "reading_timestamp",
        to_timestamp("reading_timestamp")
    )

window_spec = Window \
    .partitionBy("meter_id") \
    .orderBy("reading_timestamp", "reading_id") \
    .rowsBetween(Window.unboundedPreceding, Window.currentRow)

result_df = valid_df.withColumn(
    "cumulative_energy_kwh",
    sum("energy_kwh").over(window_spec)
)

result_df.select(
    "meter_id",
    "reading_id",
    "reading_timestamp",
    "energy_kwh",
    "cumulative_energy_kwh"
).show()

+--------+----------+-------------------+----------+---------------------+
|meter_id|reading_id|  reading_timestamp|energy_kwh|cumulative_energy_kwh|
+--------+----------+-------------------+----------+---------------------+
|   M0001| RD0083587|2026-01-01 01:15:00|     22.27|                22.27|
|   M0001| RD0086275|2026-01-01 03:30:00|      4.05|                26.32|
|   M0001| RD0088204|2026-01-01 09:45:00|     11.28|                 37.6|
|   M0001| RD0005149|2026-01-02 04:45:00|     11.76|                49.36|
|   M0001| RD0035530|2026-01-02 08:15:00|     10.06|                59.42|
|   M0001| RD0070104|2026-01-02 18:30:00|     15.83|                75.25|
|   M0001| RD0006677|2026-01-03 00:30:00|     17.06|                92.31|
|   M0001| RD0021042|2026-01-03 06:00:00|     17.14|               109.45|
|   M0001| RD0032205|2026-01-03 08:30:00|     20.46|               129.91|
|   M0001| RD0074606|2026-01-03 18:15:00|      20.4|               150.31|
|   M0001| RD0019547|2026

In [34]:
count_df1 = readings_df.count()
# Rename source_type to collection_mode to match readings_df
fixed_extra_df = extra_readings_df.withColumnRenamed("source_type", "collection_mode")
count_df2 = fixed_extra_df.count()

combined_df = readings_df.unionByName(fixed_extra_df)

combined_count = combined_df.count()

print("Rows in readings_df       :", count_df1)
print("Rows in extra_readings_df :", count_df2)
print("Rows after unionByName()  :", combined_count)
print("Expected rows             :", count_df1 + count_df2)

Rows in readings_df       : 90000
Rows in extra_readings_df : 12000
Rows after unionByName()  : 102000
Expected rows             : 102000


In [35]:
from pyspark.sql.functions import count

duplicate_df = combined_df.groupBy("reading_id") \
    .agg(count("*").alias("cnt")) \
    .filter("cnt > 1")

duplicate_df.show()
unique_df = combined_df.dropDuplicates(["reading_id"])

before_count = combined_df.count()
after_count = unique_df.count()

print("Rows before deduplication :", before_count)
print("Rows after deduplication  :", after_count)
print("Duplicates removed        :", before_count - after_count)

final_unique_count = unique_df.count()

print("Final unique reading count:", final_unique_count)

+----------+---+
|reading_id|cnt|
+----------+---+
| RD0025848|  2|
| RD0045533|  2|
| RD0064921|  2|
| RD0056575|  2|
| RD0058815|  2|
| RD0022131|  2|
| RD0029598|  2|
| RD0045710|  2|
| RD0030877|  2|
| RD0002679|  2|
| RD0011169|  2|
| RD0028736|  2|
| RD0030669|  2|
| RD0061933|  2|
| RD0015904|  2|
| RD0008188|  2|
| RD0037899|  2|
| RD0000004|  2|
| RD0025628|  2|
| RD0048911|  2|
+----------+---+
only showing top 20 rows
Rows before deduplication : 102000
Rows after deduplication  : 101300
Duplicates removed        : 700
Final unique reading count: 101300


In [37]:
from pyspark.sql.functions import count

# Rename source_type to collection_mode to match readings_df and resolve schema issues
fixed_extra_df = extra_readings_df.withColumnRenamed("source_type", "collection_mode")
combined_df = readings_df.unionByName(fixed_extra_df)

duplicate_df = combined_df.groupBy("reading_id") \
    .count() \
    .filter("count > 1")

unique_df = combined_df.dropDuplicates(["reading_id"])

print("Duplicate reading IDs:", duplicate_df.count())
print("Unique reading count:", unique_df.count())

Duplicate reading IDs: 700
Unique reading count: 101300


In [38]:
from pyspark.sql.functions import col, when, sum

pipeline_df = readings_df \
    .filter(col("reading_status") == "Valid") \
    .withColumn(
        "energy_band",
        when(col("energy_kwh") < 5, "Low")
        .when(col("energy_kwh") <= 25, "Medium")
        .otherwise("High")
    ) \
    .groupBy("energy_band") \
    .agg(
        sum("energy_kwh").alias("total_energy")
    ) \
    .orderBy(col("total_energy").desc())

# Inspect logical and physical plan
pipeline_df.explain()

# Action
pipeline_df.show()

== Physical Plan ==
AdaptiveSparkPlan isFinalPlan=false
+- Sort [total_energy#1231 DESC NULLS LAST], true, 0
   +- Exchange rangepartitioning(total_energy#1231 DESC NULLS LAST, 200), ENSURE_REQUIREMENTS, [plan_id=3096]
      +- HashAggregate(keys=[energy_band#1230], functions=[sum(energy_kwh#20)])
         +- Exchange hashpartitioning(energy_band#1230, 200), ENSURE_REQUIREMENTS, [plan_id=3093]
            +- HashAggregate(keys=[energy_band#1230], functions=[partial_sum(energy_kwh#20)])
               +- Project [energy_kwh#20, CASE WHEN (energy_kwh#20 < 5.0) THEN Low WHEN (energy_kwh#20 <= 25.0) THEN Medium ELSE High END AS energy_band#1230]
                  +- Filter (isnotnull(reading_status#24) AND (reading_status#24 = Valid))
                     +- FileScan csv [energy_kwh#20,reading_status#24] Batched: false, DataFilters: [isnotnull(reading_status#24), (reading_status#24 = Valid)], Format: CSV, Location: InMemoryFileIndex(1 paths)[file:/content/meter_readings_90000.csv], Partiti

In [39]:
import time
from pyspark.sql.functions import col

valid_df = readings_df.filter(
    col("reading_status") == "Valid"
).cache()

print("Is Cached:", valid_df.is_cached)

start = time.time()
count1 = valid_df.count()
time1 = time.time() - start

start = time.time()
count2 = valid_df.count()
time2 = time.time() - start

print("Count 1:", count1)
print("Time for first count :", time1, "seconds")

print("Count 2:", count2)
print("Time for second count:", time2, "seconds")

Is Cached: True
Count 1: 74486
Time for first count : 3.564500331878662 seconds
Count 2: 74486
Time for second count: 0.1312263011932373 seconds


In [40]:
from pyspark import StorageLevel
from pyspark.sql.functions import col

outage_df = readings_df.filter(
    col("outage_minutes") > 0
).persist(StorageLevel.MEMORY_AND_DISK)

# Action 1
outage_df.count()

# Action 2
outage_df.groupBy("reading_status").count().show()

# Release cache
outage_df.unpersist()

print("Persisted data released.")


+--------------+-----+
|reading_status|count|
+--------------+-----+
|      Rejected|  308|
|         Valid| 6345|
|     Estimated| 1000|
+--------------+-----+

Persisted data released.


In [41]:
import time
from pyspark.sql.functions import sum

agg_df = readings_df.groupBy("reading_status") \
    .agg(sum("energy_kwh").alias("total_energy"))

start = time.time()
agg_df.show()
without_cache_1 = time.time() - start

start = time.time()
agg_df.show()
without_cache_2 = time.time() - start

print("Without Cache - Run 1:", without_cache_1)
print("Without Cache - Run 2:", without_cache_2)

+--------------+------------------+
|reading_status|      total_energy|
+--------------+------------------+
|      Rejected| 68211.17999999993|
|         Valid|1401576.2899999914|
|     Estimated| 222293.2100000001|
+--------------+------------------+

+--------------+------------------+
|reading_status|      total_energy|
+--------------+------------------+
|      Rejected| 68211.17999999993|
|         Valid|1401576.2899999914|
|     Estimated| 222293.2100000001|
+--------------+------------------+

Without Cache - Run 1: 0.6492435932159424
Without Cache - Run 2: 0.773231029510498


In [42]:
import time
from pyspark.sql.functions import sum

agg_df = readings_df.groupBy("reading_status") \
    .agg(sum("energy_kwh").alias("total_energy"))

start = time.time()
agg_df.show()
without_cache_1 = time.time() - start

start = time.time()
agg_df.show()
without_cache_2 = time.time() - start

print("Without Cache - Run 1:", without_cache_1)
print("Without Cache - Run 2:", without_cache_2)

+--------------+------------------+
|reading_status|      total_energy|
+--------------+------------------+
|      Rejected| 68211.17999999993|
|         Valid|1401576.2899999914|
|     Estimated| 222293.2100000001|
+--------------+------------------+

+--------------+------------------+
|reading_status|      total_energy|
+--------------+------------------+
|      Rejected| 68211.17999999993|
|         Valid|1401576.2899999914|
|     Estimated| 222293.2100000001|
+--------------+------------------+

Without Cache - Run 1: 0.7330765724182129
Without Cache - Run 2: 0.38718271255493164


In [44]:
from pyspark.sql.functions import (
    col, to_timestamp, date_format, sum, count,
    when, dense_rank, round
)
from pyspark.sql.window import Window
from pyspark import StorageLevel

# 01
# Rename source_type to collection_mode in extra_readings_df to align with readings_df
fixed_extra_df = extra_readings_df.withColumnRenamed("source_type", "collection_mode")

combined_df = readings_df.unionByName(
    fixed_extra_df,
    allowMissingColumns=True
)

readings_clean = combined_df.dropDuplicates(["reading_id"])

# 02
readings_clean = readings_clean.na.fill({
    "energy_kwh": 0
})

# 03
joined_df = readings_clean \
    .join(meter_df, on="meter_id", how="inner") \
    .join(substation_df, on="substation_id", how="inner") \
    .join(tariff_df, on="tariff_id", how="inner")

# 04
report_base = joined_df \
    .withColumn(
        "reading_timestamp",
        to_timestamp("reading_timestamp")
    ) \
    .withColumn(
        "month",
        date_format("reading_timestamp", "yyyy-MM")
    ) \
    .withColumn(
        "billable_energy_kwh",
        when(
            col("reading_status") != "Rejected",
            col("energy_kwh")
        ).otherwise(0)
    ) \
    .withColumn(
        "energy_charge_inr",
        when(
            col("reading_status") != "Rejected",
            col("energy_kwh") * col("rate_per_kwh_inr")
        ).otherwise(0)
    ) \
    .withColumn(
        "outage_flag",
        when(col("outage_minutes") > 0, 1).otherwise(0)
    )

# 05
report_df = report_base \
    .groupBy(
        "month",
        "region",
        "site_type"
    ) \
    .agg(
        sum("billable_energy_kwh")
            .alias("total_kwh"),

        sum("energy_charge_inr")
            .alias("estimated_energy_charges"),

        sum("outage_minutes")
            .alias("total_outage_minutes"),

        count("*")
            .alias("total_readings"),

        sum("outage_flag")
            .alias("outage_readings")
    ) \
    .withColumn(
        "outage_percentage",
        round(
            col("outage_readings")
            * 100.0
            / col("total_readings"),
            2
        )
    )

# 06
rank_window = Window.partitionBy(
    "month",
    "region"
).orderBy(
    col("estimated_energy_charges").desc()
)

final_report = report_df.withColumn(
    "charge_rank",
    dense_rank().over(rank_window)
)

# 07
final_report.persist(StorageLevel.MEMORY_AND_DISK)
final_report.show(truncate=False)
final_report.count()
final_report.unpersist()

# 08
final_report \
    .repartition(2) \
    .write \
    .mode("overwrite") \
    .parquet("/output/energy_consumption_reliability_report")

print("Energy Consumption & Reliability Report Generated Successfully")

+-------+------+---------------+------------------+------------------------+--------------------+--------------+---------------+-----------------+-----------+
|month  |region|site_type      |total_kwh         |estimated_energy_charges|total_outage_minutes|total_readings|outage_readings|outage_percentage|charge_rank|
+-------+------+---------------+------------------+------------------------+--------------------+--------------+---------------+-----------------+-----------+
|2026-01|South |Industrial     |51715.76000000001 |470613.41599999997      |7058                |897           |77             |8.58             |1          |
|2026-01|South |Commercial     |36720.51999999999 |301108.26399999997      |16976               |2046          |181            |8.85             |2          |
|2026-01|South |Public Facility|15478.970000000001|103709.09899999993      |9035                |1141          |98             |8.59             |3          |
|2026-01|South |Residential    |9834.679999999